# This nb is made to generate the subsets used for the fine-tuning experiments - Hirid, Mortality, LOS, AKI

In [ ]:
import os
import copy
import logging
import json
import hashlib
import pickle
from pathlib import Path
from timeit import default_timer as timer

import gin
import pandas as pd
import polars as pl

from sklearn.model_selection import (
    StratifiedKFold,
    KFold,
    StratifiedShuffleSplit,
    ShuffleSplit,
)

from icu_benchmarks.data.preprocessor import (
    Preprocessor,
    PandasClassificationPreprocessor,
    PolarsClassificationPreprocessor,
)

from icu_benchmarks.constants import RunMode
from icu_benchmarks.run_utils import check_required_keys

from icu_benchmarks.data.constants import (
    DataSplit as Split,
    DataSegment as Segment,
    VarType as Var,
)

from icu_benchmarks.data.split_process_data import (
    check_sanitize_data,
    modality_selection,
    caching,
)

from icu_benchmarks.cross_validation import execute_repeated_cv
from icu_benchmarks.run import *



# =========================
# PATHS (your originals)
# =========================
m24_dl_subset_path = Path("/work3/s185395/YAIB/icu_benchmarks/data/preprocessed_data/Mortality24/hirid/equal")
los_dl_subset_path = Path("/work3/s185395/YAIB/icu_benchmarks/data/preprocessed_data/LengthOfStay/hirid")
aki_dl_subset_path = Path("/work3/s185395/YAIB/icu_benchmarks/data/preprocessed_data/AKI/hirid/equal")

m24_ml_subset_path = Path("/work3/s185395/YAIB/icu_benchmarks/data/preprocessed_data/ML_models/Mortality24/hirid/equal")
los_ml_subset_path = Path("/work3/s185395/YAIB/icu_benchmarks/data/preprocessed_data/ML_models/LengthOfStay/hirid")
aki_ml_subset_path = Path("/work3/s185395/YAIB/icu_benchmarks/data/preprocessed_data/ML_models/AKI/hirid/equal")

m24_raw_path = Path("/work3/s185395/YAIB-cohorts/data/mortality24/hirid")
los_raw_path = Path("/work3/s185395/YAIB-cohorts/data/los/hirid")
aki_raw_path = Path("/work3/s185395/YAIB-cohorts/data/aki/hirid")

# Old class imbalance 
m24_dl_subset_path_preserve = Path("/work3/s185395/YAIB/icu_benchmarks/data/preprocessed_data/Mortality24/hirid/preserve")
m24_ml_subset_path_preserve = Path("/work3/s185395/YAIB/icu_benchmarks/data/preprocessed_data/ML_models/Mortality24/hirid/preserve")

# =========================
# RAW PARQUET FILENAMES
# =========================
file_names = {
    "DYNAMIC": "dyn.parquet",
    "OUTCOME": "outc.parquet",
    "STATIC": "sta.parquet",
}


# =========================
# VARIABLE CONFIG
# =========================
vars_dict = {
    "GROUP": "stay_id",
    "SEQUENCE": "time",
    "LABEL": "label",
    "DYNAMIC": [
        "alb", "alp", "alt", "ast", "be", "bicar", "bili", "bili_dir", "bnd", "bun", "ca", "cai", "ck",
        "ckmb", "cl", "crea", "crp", "dbp", "fgn", "fio2", "glu", "hgb", "hr", "inr_pt", "k", "lact",
        "lymph", "map", "mch", "mchc", "mcv", "methb", "mg", "na", "neut", "o2sat", "pco2", "ph", "phos",
        "plt", "po2", "ptt", "resp", "sbp", "temp", "tnt", "urine", "wbc"
    ],
    "STATIC": ["age", "sex", "height", "weight"],
}

modality_mapping = {
    "DYNAMIC": [
        "alb", "alp", "alt", "ast", "be", "bicar", "bili", "bili_dir", "bnd", "bun", "ca", "cai", "ck",
        "ckmb", "cl", "crea", "crp", "dbp", "fgn", "fio2", "glu", "hgb", "hr", "inr_pt", "k", "lact",
        "lymph", "map", "mch", "mchc", "mcv", "methb", "mg", "na", "neut", "o2sat", "pco2", "ph", "phos",
        "plt", "po2", "ptt", "resp", "sbp", "temp", "tnt", "urine", "wbc"
    ],
    "STATIC": ["age", "sex", "height", "weight"],
}


# =========================
# GIN CONFIGS (your originals)
# =========================
gin.clear_config()
gin.parse_config_file("/work3/s185395/YAIB/configs/tasks/BinaryClassification.gin")
gin.parse_config_file("/work3/s185395/YAIB/configs/prediction_models/common/MLCommon.gin")
gin.bind_parameter("execute_repeated_cv.cv_repetitions", None)

# -----------------------------
# SHIFT LOGIC (same as subsets)
# -----------------------------
SHIFT = 1_000_000_000_000


def build_id_map(requested_ids: list[int] | set[int], shift: int = SHIFT) -> pl.DataFrame:
    """
    requested_ids can include base ids and shifted ids.
    Returns mapping:
      new_stay_id  base_stay_id
    """
    return (
        pl.DataFrame({"new_stay_id": list(requested_ids)})
        .with_columns((pl.col("new_stay_id") % shift).alias("base_stay_id"))
    )


def fetch_and_expand_segment(
    df_raw: pl.DataFrame,
    id_map: pl.DataFrame,
    raw_id_col: str = "stay_id",
) -> pl.DataFrame:
    if df_raw.height == 0 or id_map.height == 0:
        return df_raw.head(0)

    # Cast for safe joins
    df_raw2 = df_raw.with_columns(pl.col(raw_id_col).cast(pl.Int64))
    id_map2 = id_map.with_columns(
        pl.col("new_stay_id").cast(pl.Int64),
        pl.col("base_stay_id").cast(pl.Int64),
    )

    # Filter raw to needed base ids
    base_ids = id_map2.select("base_stay_id").unique()
    df_fetched = df_raw2.join(base_ids, left_on=raw_id_col, right_on="base_stay_id", how="inner")

    # Expand/duplicate by joining on base id
    joined = df_fetched.join(id_map2, left_on=raw_id_col, right_on="base_stay_id", how="inner")

    # Build output deterministically:
    # keep all original raw columns EXCEPT the base stay_id
    raw_cols = [c for c in df_raw2.columns if c != raw_id_col]

    out = joined.select(
        [pl.col("new_stay_id").alias(raw_id_col)] + [pl.col(c) for c in raw_cols]
    )

    return out

def make_split_from_ids_with_shift_support(
    data: dict[str, pl.DataFrame],     # keys like "STATIC", "DYNAMIC", "OUTCOME"
    train_ids: list[int] | set[int],
    val_ids: list[int] | set[int],
    test_ids: list[int] | set[int],
    id_col: str = "stay_id",
    shift: int = SHIFT,
    check_disjoint: bool = True,
    verbose: bool = True,
) -> dict[str, dict[str, pl.DataFrame]]:
    """
    Returns:
      {
        "train": {"STATIC": df, "DYNAMIC": df, "OUTCOME": df, ...},
        "val":   {...},
        "test":  {...}
      }
    with shifted-id support.
    """
    train_ids = list(train_ids)
    val_ids = list(val_ids)
    test_ids = list(test_ids)

    if check_disjoint:
        st, sv, se = set(train_ids), set(val_ids), set(test_ids)
        if (st & sv) or (st & se) or (sv & se):
            raise ValueError("ID lists overlap across splits.")

    split_to_ids = {"train": train_ids, "val": val_ids, "test": test_ids}
    out = {split: {} for split in split_to_ids}

    # Precompute id maps per split
    split_to_map = {split: build_id_map(ids, shift=shift) for split, ids in split_to_ids.items()}

    for seg_name, df_raw in data.items():
        if id_col not in df_raw.columns:
            raise KeyError(f"Column '{id_col}' not found in segment {seg_name}. Columns: {df_raw.columns}")

        for split, id_map in split_to_map.items():
            out[split][seg_name] = fetch_and_expand_segment(df_raw, id_map, raw_id_col=id_col)

    if verbose:
        for split in ["train", "val", "test"]:
            for seg_name, df in out[split].items():
                n_unique = df.select(pl.col(id_col).n_unique()).item() if df.height else 0
                print(f"{split}/{seg_name}: rows={df.height}, unique_{id_col}={n_unique}")

    return out


def preprocess_data_modified(
    data_dir: Path,
    file_names: dict[str, str] = gin.REQUIRED,      # {"DYNAMIC":"dyn.parquet","OUTCOME":"outc.parquet","STATIC":"sta.parquet"}
    preprocessor: Preprocessor = PolarsClassificationPreprocessor,
    use_static: bool = True,
    vars: dict[str, str | list[str]] = gin.REQUIRED,
    modality_mapping: dict[str, list[str]] = {},
    selected_modalities: list[str] | str = "all",
    seed: int = 42,
    debug: bool = False,
    cv_repetitions: int = 5,
    repetition_index: int = 0,
    cv_folds: int = 5,
    train_size: int = None,
    load_cache: bool = False,
    generate_cache: bool = False,
    fold_index: int = 0,
    pretrained_imputation_model: str = None,
    complete_train: bool = False,
    runmode: RunMode = RunMode.classification,
    label: str = None,
    required_var_types=["GROUP", "SEQUENCE", "LABEL"],
    required_segments=[Segment.static, Segment.dynamic, Segment.outcome],
    add_age_binary=None,
    train_ids: list[int] | set[int] = None,
    val_ids: list[int] | set[int] = None,
    test_ids: list[int] | set[int] = None,
    shift: int = SHIFT,
) -> dict[str, dict[str, pl.DataFrame]] | dict[str, dict[str, pd.DataFrame]]:
    """
    Loads raw parquet, sanitizes, splits using provided IDs (supports shifted IDs),
    then applies the preprocessor.
    """

    if train_ids is None or val_ids is None or test_ids is None:
        raise ValueError("train_ids, val_ids, test_ids must be provided.")

    cache_dir = data_dir / "cache"

    check_required_keys(vars, required_var_types)
    # file_names are strings here; required_segments check in original code expects Segment enums.
    # We'll just ensure needed keys exist in file_names:
    for k in ["OUTCOME", "DYNAMIC"] + (["STATIC"] if use_static else []):
        if k not in file_names:
            raise KeyError(f"file_names missing required key '{k}'. Got keys: {list(file_names.keys())}")

    if not use_static:
        file_names = dict(file_names)
        file_names.pop("STATIC", None)
        vars = dict(vars)
        vars.pop(Segment.static, None)
        vars.pop("STATIC", None)

    # Handle multi-label case like original
    if isinstance(vars.get(Var.label), list) and len(vars[Var.label]) > 1:
        if label is not None:
            vars[Var.label] = [label]
        else:
            logging.debug(f"Multiple labels found and no value provided. Using first label: {vars[Var.label]}")
            vars[Var.label] = vars[Var.label][0]
        logging.info(f"Using label: {vars[Var.label]}")

    if not vars.get(Var.label):
        raise ValueError("No label selected after filtering.")

    dumped_file_names = json.dumps(file_names, sort_keys=True)
    dumped_vars = json.dumps(vars, sort_keys=True)

    cache_filename = f"s_{seed}_r_{repetition_index}_f_{fold_index}_t_{train_size}_d_{debug}"
    logging.info(f"Using preprocessor: {preprocessor.__name__}")

    vars_to_exclude = (
        modality_mapping.get("cat_clinical_notes", []) + modality_mapping.get("cat_med_embeddings_map", [])
        if modality_mapping.get("cat_clinical_notes") is not None and modality_mapping.get("cat_med_embeddings_map") is not None
        else None
    )

    preprocessor = preprocessor(
        use_static_features=use_static,
        save_cache=data_dir / "preproc" / (cache_filename + "_recipe"),
        vars_to_exclude=vars_to_exclude,
    )
    if isinstance(preprocessor, PandasClassificationPreprocessor):
        preprocessor.set_imputation_model(pretrained_imputation_model)

    hash_config = hashlib.md5(f"{preprocessor.to_cache_string()}{dumped_file_names}{dumped_vars}".encode("utf-8"))
    cache_filename += f"_{hash_config.hexdigest()}"
    cache_file = cache_dir / cache_filename

    if load_cache and cache_file.exists():
        with open(cache_file, "rb") as f:
            logging.info(f"Loading cached data from {cache_file}.")
            return pickle.load(f)

    # ----------------------------
    # Load raw data (string keys)
    # ----------------------------
    logging.info(f"Loading data from directory {data_dir.absolute()}")
    data = {
        seg: pl.read_parquet(data_dir / fname)
        for seg, fname in file_names.items()
        if os.path.exists(data_dir / fname)
    }
    logging.info(f"Loaded segments: {list(data.keys())}")

    # ----------------------------
    # Missing check (BASE IDs only)
    # ----------------------------
    if "OUTCOME" in data:
        raw_outcome = data["OUTCOME"]
        id_map_train = build_id_map(train_ids, shift=shift)
        base_train_ids = set(id_map_train.select("base_stay_id").unique().to_series().to_list())
        present_base = set(raw_outcome.select("stay_id").unique().to_series().to_list())
        missing_base = base_train_ids - present_base
        print("Missing BASE ids in raw OUTCOME:", len(missing_base), list(sorted(missing_base))[:20])

    # Sanitize (your existing function)
    data = check_sanitize_data(data, vars)

    # Modality selection (optional)
    if len(modality_mapping) > 0:
        if selected_modalities not in [None, "all", ["all"]]:
            data, vars = modality_selection(data, modality_mapping, selected_modalities, vars)
        else:
            logging.info("Selecting all modalities.")

    # ----------------------------
    # Split with shifted-id support
    # ----------------------------
    logging.info("Generating splits (shift-aware).")
    data = make_split_from_ids_with_shift_support(
        data=data,
        train_ids=train_ids,
        val_ids=val_ids,
        test_ids=test_ids,
        id_col="stay_id",
        shift=shift,
        check_disjoint=True,
        verbose=True,
    )

    # ----------------------------
    # Sanity checks BEFORE preprocessing
    # ----------------------------
    id_map = build_id_map(train_ids, shift=shift)
    print("requested:", len(list(train_ids)))
    print("unique new:", id_map.select(pl.col("new_stay_id").n_unique()).item())
    print("unique base:", id_map.select(pl.col("base_stay_id").n_unique()).item())
    print(id_map.group_by("base_stay_id").len().sort("len", descending=True).head(10))

    outcome_train = data["train"].get("OUTCOME")
    if outcome_train is not None:
        print(
            outcome_train.select(pl.col("stay_id").n_unique()).item(),
            "unique stay_id in expanded train OUTCOME"
        )

    # ----------------------------
    # Apply preprocessing
    # ----------------------------
    start = timer()
    data = preprocessor.apply(data, vars)
    end = timer()
    logging.info(f"Preprocessing took {end - start:.2f} seconds.")

    # Optional null/NaN fill (your original loop)
    for split_dict in data.values():
        for seg_key, df in split_dict.items():
            if isinstance(df, pl.DataFrame):
                split_dict[seg_key] = (
                    df.fill_null(strategy="zero")
                      .fill_nan(0)
                )

    # Cache
    if generate_cache:
        caching(cache_dir, cache_file, data, load_cache)
    else:
        logging.info("Cache will not be saved.")

    logging.info("Finished preprocessing.")
    return data


In [ ]:
# =========================
# RUN ONE EXAMPLE: subset size 100
# =========================
# Choose task you want to test:
subset_path = m24_dl_subset_path # change to m24_dl_subset_path / los_dl_subset_path as needed
raw_path    = m24_raw_path         # change accordingly
out_base    = m24_ml_subset_path # output folder

subset_size = "9000"
seed_str    = "42"   # pick one that exists in your preprocessed subset folders

splits = {
    "train": "train_OUTCOME.parquet",
    "val":   "val_OUTCOME.parquet",
    "test":  "test_OUTCOME.parquet",
}

# 1) Read stay_ids from your ALREADY-CREATED subset outcome parquets
stay_ids = {}
for split, fname in splits.items():
    df_ids = pd.read_parquet(subset_path / f"{subset_size}_{seed_str}" / fname, columns=["stay_id"])
    stay_ids[split] = df_ids["stay_id"].unique().tolist()

train_stay_ids = stay_ids["train"]
val_stay_ids   = stay_ids["val"]
test_stay_ids  = stay_ids["test"]

print(f"Train: {len(train_stay_ids)}")
print(f"Val:   {len(val_stay_ids)}")
print(f"Test:  {len(test_stay_ids)}")

# 2) Run preprocessing from RAW using shift-aware splitting
data = preprocess_data_modified(
    data_dir=raw_path,
    vars=vars_dict,
    file_names=file_names,
    seed=2222,
    generate_cache=False,
    load_cache=False,
    debug=False,
    use_static=True,
    runmode=RunMode.classification,
    train_ids=train_stay_ids,
    val_ids=val_stay_ids,
    test_ids=test_stay_ids,
)

# 3) Save results to output folder (same style you used)
out_dir = out_base / f"{subset_size}_{seed_str}"
out_dir.mkdir(parents=True, exist_ok=True)

for split, split_dict in data.items():
    for seg_key, df in split_dict.items():
        seg_name = seg_key.upper() if isinstance(seg_key, str) else str(seg_key).upper()
        if isinstance(df, pl.DataFrame):
            df.write_parquet(out_dir / f"{split}_{seg_name}.parquet")
        else:
            df.to_parquet(out_dir / f"{split}_{seg_name}.parquet", index=False)

print(f"Saved to: {out_dir}")


In [ ]:
subset_path = m24_dl_subset_path
raw_path    = m24_raw_path
out_base    = m24_ml_subset_path

subset_sizes = ["100","500","1000","2000","3000","5000","7000","9000","9506"]
seeds = ["42","84","126","168","210"]

splits = {
    "train": "train_OUTCOME.parquet",
    "val":   "val_OUTCOME.parquet",
    "test":  "test_OUTCOME.parquet",
}

def save_preprocessed_splits(data, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    for split, split_dict in data.items():
        for seg_key, df in split_dict.items():
            seg_name = seg_key.upper() if isinstance(seg_key, str) else str(seg_key).upper()
            if isinstance(df, pl.DataFrame):
                df.write_parquet(out_dir / f"{split}_{seg_name}.parquet")
            else:
                df.to_parquet(out_dir / f"{split}_{seg_name}.parquet", index=False)

for subset_size in subset_sizes:
    for seed_str in seeds:
        subset_dir = subset_path / f"{subset_size}_{seed_str}"
        if not subset_dir.exists():
            print(f"❌ Missing subset folder, skipping: {subset_dir}")
            continue

        # 1) Read stay_ids from existing subset outcome parquets
        stay_ids = {}
        missing_split_files = []
        for split, fname in splits.items():
            p = subset_dir / fname
            if not p.exists():
                missing_split_files.append(str(p))
                continue
            df_ids = pd.read_parquet(p, columns=["stay_id"])
            stay_ids[split] = df_ids["stay_id"].unique().tolist()

        if missing_split_files:
            print(f"❌ Missing split files for {subset_size}_{seed_str}, skipping:")
            for p in missing_split_files:
                print("   ", p)
            continue

        train_stay_ids = stay_ids["train"]
        val_stay_ids   = stay_ids["val"]
        test_stay_ids  = stay_ids["test"]

        print("\n" + "="*80)
        print(f"Running Mortality24 subset={subset_size} seed={seed_str}")
        print(f"Train: {len(train_stay_ids)} | Val: {len(val_stay_ids)} | Test: {len(test_stay_ids)}")

        # 2) Preprocess from RAW using shift-aware splitting
        data = preprocess_data_modified(
            data_dir=raw_path,
            vars=vars_dict,
            file_names=file_names,
            seed=2222,
            generate_cache=False,
            load_cache=False,
            debug=False,
            use_static=True,
            runmode=RunMode.classification,
            train_ids=train_stay_ids,
            val_ids=val_stay_ids,
            test_ids=test_stay_ids,
        )

        # 3) Save results
        out_dir = out_base / f"{subset_size}_{seed_str}"
        save_preprocessed_splits(data, out_dir)
        print(f"✅ Saved to: {out_dir}")


In [ ]:
import polars as pl
from pathlib import Path

def load_ids(parquet_path: Path) -> set[int]:
    # read only stay_id, ensure int64, return as python set
    s = (
        pl.read_parquet(parquet_path, columns=["stay_id"])
        .select(pl.col("stay_id").cast(pl.Int64))
        .to_series()
        .to_list()
    )
    return set(s)

def compare_two(a_path: Path, b_path: Path, name_a="A", name_b="B"):
    a = load_ids(a_path)
    b = load_ids(b_path)

    inter = a & b
    only_a = a - b
    only_b = b - a

    print("\n" + "="*80)
    print(f"{name_a}: {a_path}")
    print(f"{name_b}: {b_path}")
    print("-"*80)
    print(f"{name_a} unique stay_ids: {len(a)}")
    print(f"{name_b} unique stay_ids: {len(b)}")
    print(f"Intersection:          {len(inter)}")
    print(f"Only in {name_a}:       {len(only_a)}")
    print(f"Only in {name_b}:       {len(only_b)}")

    # show a few examples
    if only_a:
        print(f"\nExamples only in {name_a} (up to 20): {sorted(list(only_a))[:20]}")
    if only_b:
        print(f"Examples only in {name_b} (up to 20): {sorted(list(only_b))[:20]}")

# ---- Paths you gave ----
ml_base = Path("/work3/s185395/YAIB/icu_benchmarks/data/preprocessed_data/ML_models/Mortality24/hirid/equal")
dl_base = Path("/work3/s185395/YAIB/icu_benchmarks/data/preprocessed_data/Mortality24/hirid/equal")

# 100_42
compare_two(
    a_path=ml_base / "100_42" / "train_FEATURES.parquet",
    b_path=dl_base / "100_42" / "train_FEATURES.parquet",
    name_a="ML_models",
    name_b="DL_subset",
)

# 7000_42
compare_two(
    a_path=ml_base / "9000_42" / "train_FEATURES.parquet",
    b_path=dl_base / "9000_42" / "train_FEATURES.parquet",
    name_a="ML_models",
    name_b="DL_subset",
)


In [ ]:
import polars as pl
from pathlib import Path

def inspect_parquet(path: Path, n: int = 5):
    print("\n" + "=" * 80)
    print(f"File: {path}")

    if not path.exists():
        print("❌ File does not exist")
        return

    df = pl.read_parquet(path)

    print(f"Shape: {df.shape}")
    print("\nColumns:")
    print(df.columns)

    print(f"\nHead ({n} rows):")
    print(df.head(n))


base_ml = Path("/work3/s185395/YAIB/icu_benchmarks/data/preprocessed_data/ML_models/Mortality24/hirid/equal")

# 100_42
inspect_parquet(base_ml / "100_42" / "train_FEATURES.parquet")

# 9000_42
inspect_parquet(base_ml / "9000_42" / "train_FEATURES.parquet")
